# Tonality Features (IEMOCAP)

This notebook extracts chroma and tonnetz summaries.
Each utterance becomes one training row for downstream SER models.

In [1]:
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import librosa
import numpy as np
import pandas as pd


In [2]:
# Configuration
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not locate repository root (missing pyproject.toml).")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from feature_extraction.common import machine_name_from_env, resolve_thread_workers

MACHINE_NAME = machine_name_from_env()
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "tonality"
OUT_FILE = "tonality_features.csv"

# Audio + feature params
TARGET_SR = 16_000
HOP_LENGTH = 512
N_FFT = 2048
STATS = ("mean", "std", "p10", "p50", "p90")

EXCLUDED_EMOTIONS = {"sur", "fea", "oth", "dis"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


WindowsPath('F:/Speech-Emotion-Recognition/extracted_features/tonality/tonality_features.csv')

In [3]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def summarize_curve(prefix: str, values: np.ndarray, stats: tuple[str, ...]) -> dict[str, float]:
    # Summary stats ignoring NaNs
    vec = np.asarray(values, dtype=float).ravel()
    vec = vec[~np.isnan(vec)]
    if vec.size == 0:
        return {f"{prefix}_{stat}": float("nan") for stat in stats}

    summary: dict[str, float] = {}
    for stat in stats:
        key = f"{prefix}_{stat}"
        if stat == "mean":
            summary[key] = float(np.mean(vec))
        elif stat == "std":
            summary[key] = float(np.std(vec))
        elif stat.startswith("p") and stat[1:].isdigit():
            percentile = int(stat[1:])
            summary[key] = float(np.percentile(vec, percentile))
        else:
            raise ValueError(f"Unknown stat: {stat}")
    return summary


def extract_tonality_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=sr,
        hop_length=HOP_LENGTH,
        n_fft=N_FFT,
    )
    harmonic = librosa.effects.harmonic(audio)
    tonnetz = librosa.feature.tonnetz(y=harmonic, sr=sr)

    features: dict[str, float] = {}
    for idx, curve in enumerate(chroma):
        features.update(summarize_curve(f"chroma_{idx:02d}", curve, STATS))
    for idx, curve in enumerate(tonnetz):
        features.update(summarize_curve(f"tonnetz_{idx:02d}", curve, STATS))
    return features


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: keep xxx, exclude selected classes, and enforce agreement for labeled classes.
# This keeps unlabeled (xxx) examples while dropping sur/fea/oth/dis.
df = df[~df["emotion"].isin(EXCLUDED_EMOTIONS)].copy()
df = df[(df["emotion"] == "xxx") | (df["agreement"] > 0)].copy()
df.shape


(9887, 7)

In [5]:
CPU_COUNT = os.cpu_count() or 1
COMPUTE_DEVICE = "cpu"  # Force CPU for this CPU-bound extractor
NUM_WORKERS = resolve_thread_workers(MACHINE_NAME)
PROGRESS_MIN_INTERVAL = 1.0

print(
    f"Compute device: {COMPUTE_DEVICE} | extractor_backend=cpu | "
    f"machine={MACHINE_NAME} | workers={NUM_WORKERS}"
)


def process_row(row: dict[str, object]) -> tuple[dict[str, float | str | int] | None, str | None]:
    rel_path = str(row["path"])
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        return None, str(audio_path)

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_tonality_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": rel_path,
        "session": int(row["session"]),
        "method": str(row["method"]),
        "gender": str(row["gender"]),
        "emotion": str(row["emotion"]),
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    return record, None


rows: list[dict[str, float | str | int]] = []
missing: list[str] = []
records = df.to_dict(orient="records")

if NUM_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        mapped = executor.map(process_row, records)
        for record, missing_path in tqdm(mapped, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
            if missing_path is not None:
                missing.append(missing_path)
                continue
            if record is not None:
                rows.append(record)
else:
    for record in tqdm(records, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
        row_result, missing_path = process_row(record)
        if missing_path is not None:
            missing.append(missing_path)
            continue
        if row_result is not None:
            rows.append(row_result)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Workers used: {NUM_WORKERS} (cpu_count={CPU_COUNT})")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape


Compute device: cpu | extractor_backend=cpu | workers=14


Extracting:   0%|          | 0/9887 [00:00<?, ?file/s]

F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=455
  warnings.warn(
F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=498
  warnings.warn(
F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=480
  warnings.warn(
F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=395
  warnings.warn(
F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=343
  warnings.warn(
F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=446
  warning

F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=320
  warnings.warn(


F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=415
  warnings.warn(


...

F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=316
  warnings.warn(


F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=256
  warnings.warn(


F:\Speech-Emotion-Recognition\.venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=302
  warnings.warn(


Saved: F:\Speech-Emotion-Recognition\extracted_features\tonality\tonality_features.csv
Workers used: 14 (cpu_count=16)


(9887, 98)